# ML-07 — Baseline Action Score and Top-10 Review

**Author:** Mehak Zahra  
**Lane locked:** Refresh / Content Opportunity Scoring  
**Question:** Which measurable content pages should an editor review first?

This notebook uses the 30,000-page public-safe starter slice. The label is used only after ranking to evaluate the frozen rule. It never contributes to the score.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
            if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists())
DATA_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
OUTPUT_DIR = ROOT / 'work' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)
print(f'Loaded {len(df):,} public-safe page rows; base rate = {df.is_declining_label.mean():.3f}')

Loaded 30,000 public-safe page rows; base rate = 0.542


## 1. Check two signals before writing the rule

The tables print `n`; small buckets are not allowed to masquerade as strong evidence. Decline rate is an observed diagnostic outcome, not a causal effect.

### Signal 1 — staleness behind refresh flags

**Verdict: MIXED.** The 90–179 day bucket has a higher observed decline rate than the freshest bucket, but the relationship does not increase consistently and the oldest buckets are tiny. Staleness alone is not reliable enough for my rule.

In [2]:
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'], [-1, 89, 179, 364, np.inf],
    labels=['0-89', '90-179', '180-364', '365+'])
staleness = (df.groupby('staleness_bucket', observed=True)
    .agg(n=('content_id', 'size'),
         observed_decline_rate=('is_declining_label', 'mean'),
         median_impressions=('impressions_90d', 'median'))
    .reset_index())
staleness['observed_decline_rate'] = staleness['observed_decline_rate'].round(3)
staleness['median_impressions'] = staleness['median_impressions'].round().astype(int)
print(staleness.to_string(index=False))
print('Verdict: MIXED')

staleness_bucket     n  observed_decline_rate  median_impressions
            0-89 20655                  0.512                 472
          90-179  9171                  0.611                1692
         180-364   169                  0.467                  16
            365+     5                  0.600                   2
Verdict: MIXED


### Signal 2 — CTR versus position behind CTR-fix logic

I restrict the check to measurable pages with at least 100 impressions and valid average position 1–20. CTR is stored as percentage points (`0.50` means 0.50%).

**Verdict: CONFIRMED.** Observed decline falls steadily from 74.4% in the lowest CTR quartile to 52.1% in the highest. This supports using a low-CTR opportunity component for review prioritization, but it does not prove that rewriting metadata will cause recovery.

In [3]:
ctr_slice = df.loc[
    (df['avg_position'] > 0) & (df['avg_position'] <= 20) &
    (df['impressions_90d'] >= 100)].copy()
ctr_slice['ctr_bucket'] = pd.qcut(
    ctr_slice['ctr'].rank(method='first'), 4,
    labels=['lowest', 'low-mid', 'high-mid', 'highest'])
ctr_table = (ctr_slice.groupby('ctr_bucket', observed=True)
    .agg(n=('content_id', 'size'),
         observed_decline_rate=('is_declining_label', 'mean'),
         median_ctr_pct=('ctr', 'median'))
    .reset_index())
ctr_table['observed_decline_rate'] = ctr_table['observed_decline_rate'].round(3)
ctr_table['median_ctr_pct'] = ctr_table['median_ctr_pct'].round(2)
print(ctr_table.to_string(index=False))
print('Verdict: CONFIRMED')

ctr_bucket    n  observed_decline_rate  median_ctr_pct
    lowest 3773                  0.744            0.00
   low-mid 3773                  0.626            0.13
  high-mid 3772                  0.586            0.28
   highest 3773                  0.521            0.66
Verdict: CONFIRMED


## 2. Encode and freeze one transparent rule

Plain-language rule: prioritize a page when it has meaningful visibility, ranks between positions 1 and 20, and has CTR below 0.50%. The score multiplies three 0–1 components, so a page must have all three forms of opportunity to rise.

`score = visibility percentile × position opportunity × CTR gap × eligibility`

- Exactly one reason code: `visible_low_ctr_review`
- Exactly one action label: `review_title_and_snippet`
- Eligibility: at least 100 trailing-90-day impressions

No future-window or label-derived input enters this calculation.

In [4]:
visibility = np.log1p(df['impressions_90d']).rank(method='average', pct=True)
position_opportunity = (((20 - df['avg_position']) / 19).clip(0, 1)
                        * df['avg_position'].between(0.000001, 20).astype(int))
ctr_gap = ((0.50 - df['ctr']) / 0.50).clip(0, 1)
eligible = (df['impressions_90d'] >= 100).astype(int)

df['baseline_score'] = (visibility * position_opportunity * ctr_gap * eligible).clip(0, 1)
df['reason_code'] = 'visible_low_ctr_review'
df['action_label'] = 'review_title_and_snippet'
df['baseline_rank'] = df['baseline_score'].rank(method='first', ascending=False).astype(int)

queue_columns = ['content_id', 'client_id', 'baseline_rank', 'baseline_score',
                 'reason_code', 'action_label', 'impressions_90d', 'avg_position',
                 'ctr', 'is_declining_label']
queue = df[queue_columns].sort_values('baseline_rank').reset_index(drop=True)
csv_path = OUTPUT_DIR / 'baseline_action_score.csv'
queue.to_csv(csv_path, index=False)

base_rate = queue['is_declining_label'].mean()
p10 = queue.head(10)['is_declining_label'].mean()
p50 = queue.head(50)['is_declining_label'].mean()
print(queue.head(10)[['baseline_rank', 'content_id', 'baseline_score',
                      'impressions_90d', 'avg_position', 'ctr']].to_string(index=False))
print(f'Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv')
print(f'Base rate: {base_rate:.3f} | Precision@10: {p10:.3f} | Precision@50: {p50:.3f}')

 baseline_rank           content_id  baseline_score  impressions_90d  avg_position  ctr
             1 content_4a6607efcb46        0.915137           128068           2.2 0.01
             2 content_8451fc6f034d        0.875217           272144           2.3 0.03
             3 content_d225ec9f3d46        0.861480            26470           0.7 0.05
             4 content_954cc45bd437        0.825201            15439           1.5 0.04
             5 content_339b357d04c7        0.824903            46879           3.7 0.01
             6 content_0022a6b4290f        0.819888            29747           1.2 0.07
             7 content_cbdf5a78dcd0        0.816032            14830           2.4 0.02
             8 content_f4e210ee0c27        0.812924            24784           1.6 0.06
             9 content_6973328a6bb8        0.796303            16512           1.0 0.07
            10 content_998f6f88784c        0.790568            12053           2.6 0.02
Wrote 30,000 rows to work/output

## 3. Top-10 skeptical review

Each row has the same frozen action and reason code. The review explains the page-specific measurements and what could invalidate the recommendation. `is_declining_label` is displayed only for retrospective evaluation.

In [5]:
top10 = queue.head(10).copy()
top10['why_it_is_here'] = top10.apply(
    lambda r: (f"{int(r.impressions_90d):,} impressions, position {r.avg_position:.1f}, "
               f"CTR {r.ctr:.2f}% produce score {r.baseline_score:.3f}."), axis=1)
top10['what_would_make_it_wrong'] = top10.apply(
    lambda r: ('A branded/navigation query mix, a GSC reporting mismatch, or a rich result '
               'could make this CTR normal for the page.') if r.avg_position <= 3 else
              ('The snippet may already match intent; mixed query intent or position averaging '
               'could make the apparent CTR gap misleading.'), axis=1)
review = top10[['baseline_rank', 'content_id', 'action_label', 'why_it_is_here',
                'what_would_make_it_wrong', 'is_declining_label']]
print(review.to_string(index=False))

 baseline_rank           content_id             action_label                                                    why_it_is_here                                                                                                   what_would_make_it_wrong  is_declining_label
             1 content_4a6607efcb46 review_title_and_snippet 128,068 impressions, position 2.2, CTR 0.01% produce score 0.915.        A branded/navigation query mix, a GSC reporting mismatch, or a rich result could make this CTR normal for the page.                   0
             2 content_8451fc6f034d review_title_and_snippet 272,144 impressions, position 2.3, CTR 0.03% produce score 0.875.        A branded/navigation query mix, a GSC reporting mismatch, or a rich result could make this CTR normal for the page.                   0
             3 content_d225ec9f3d46 review_title_and_snippet  26,470 impressions, position 0.7, CTR 0.05% produce score 0.861.        A branded/navigation query mix, a GSC reporting mismatch

## 4. Weak picks and leakage check

Four of the top ten are retrospective false positives (`is_declining_label = 0`). They are useful warnings: extremely low CTR at a strong average position can reflect branded/navigation intent, mixed queries, SERP features, or measurement behavior—not necessarily a metadata problem.

The rule's Precision@10 is 0.60, while Precision@50 is 0.76 and the full-slice base rate is 0.54. The baseline is frozen now; Week 5 must compare against it on the same slice and split.

Leakage boundary: `trend_direction`, `trend_pct`, `is_declining_label`, and all last/previous-30-day impression, click, and session fields are excluded from `SCORE_INPUTS`. IDs are used only for identity/grouping.

In [6]:
SCORE_INPUTS = ['impressions_90d', 'avg_position', 'ctr']
FORBIDDEN = {'trend_direction', 'trend_pct', 'is_declining_label',
             'impressions_last_30d', 'impressions_prev_30d',
             'clicks_last_30d', 'clicks_prev_30d',
             'sessions_last_30d', 'sessions_prev_30d'}
assert set(SCORE_INPUTS).isdisjoint(FORBIDDEN)
assert queue['reason_code'].nunique() == 1
assert queue['action_label'].nunique() == 1

metrics = {
    'assignment': 'ML-07', 'author': 'Mehak Zahra',
    'lane': 'Refresh / Content Opportunity Scoring', 'rows': len(queue),
    'base_rate': round(float(base_rate), 6),
    'precision_at_10': round(float(p10), 6),
    'precision_at_50': round(float(p50), 6),
    'reason_code': queue['reason_code'].iloc[0],
    'action_label': queue['action_label'].iloc[0],
    'score_inputs': SCORE_INPUTS,
    'excluded_from_score': sorted(FORBIDDEN),
    'signal_verdicts': {'staleness': 'MIXED', 'ctr_vs_position': 'CONFIRMED'},
}
(OUTPUT_DIR / 'baseline_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
print('Leakage check passed; wrote work/outputs/baseline_metrics.json')

Leakage check passed; wrote work/outputs/baseline_metrics.json


## 5. Self-check

- [x] Lane is confirmed and locked.
- [x] Exactly two signal bucket tables show `n` and one-word verdicts.
- [x] At least one test is tied to a real FlyRank flag; both are.
- [x] One transparent rule produces a score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] All ten top rows include action, why, and what would make the recommendation wrong.
- [x] No future-window or label-derived input contributes to the score.
- [x] Notebook outputs are executed and visible; the reproducible metrics JSON is committed.